# Bridging-Based Ranking and its Failure Modes — A Hands-On Tutorial

**Course context.** Week 5 lecture (`docs/05.md` §3 and §4): X's Community Notes uses a bridging matrix-factorisation algorithm — only notes endorsed by raters from *both* sides of an inferred polarity axis get displayed. Two papers stress-test this rule on real CN data:

- **Chuai et al. 2026** — *consensus stability* — 30.2 % of "helpful" notes lose that status after display.
- **Nudo et al. 2026** — *hyperactive minority* — rating activity is heavy-tailed and politically selective.

**What this notebook does.** Build a small synthetic Community Notes simulation: 200 raters with hidden polarity rate 60 notes with hidden quality + polarity. Implement the bridging matrix factorisation from scratch in ~30 lines. Then run two experiments:

- **§5 (Chuai)** — start with a balanced rater pool, simulate "display" by adding polarised post-display raters, refit. Watch helpful-status erode.
- **§6 (Nudo)** — generate ratings with heavy-tailed *and politically asymmetric* activity. Show empirically that the effective rater pool is non-representative.

**Prerequisites.** `numpy`, `scipy`, `matplotlib`. No API key, no internet. Runs in under a minute.

```bash
pip install numpy scipy matplotlib
```


## §1 — The bridging algorithm in one paragraph

X's Community Notes uses a matrix factorisation on rating data. Each rater $u$ and each note $n$ get two latent scalars:

- **polarity** $\gamma_u, \delta_n$ — a single ideology axis estimated from rating patterns,
- **helpfulness intercept** $\alpha_u, \beta_n$ — an axis-orthogonal baseline (rater bias / note quality).

The model: each observed rating $r_{u,n}$ is fit as
$$r_{u,n} \;\approx\; \alpha_u + \beta_n + \gamma_u \cdot \delta_n.$$

A note is displayed only if its **helpfulness intercept** $\beta_n$ exceeds threshold. Crucially, $\beta_n$ is *high only when raters from both poles of $\gamma$ rate the note highly* — because the polarity term $\gamma_u \cdot \delta_n$ already absorbs partisan agreement. So the threshold rule encodes a *procedural* value: cross-partisan endorsement.


## §2 — A synthetic Community Notes setup

Ground truth (hidden from the algorithm):

- **200 raters**, each with a polarity drawn from $\mathcal N(0, 0.5)$. Mostly moderate, some extreme.
- **60 notes**, each with a polarity uniform on $[-1, 1]$ and a quality from $\mathcal N(0, 0.5)$.
- **Each rater rates ~30 notes** (sampled randomly). The rating they give is

  $$r_{u,n} \;=\; \text{quality}_n \;+\; \text{polarity}_u \cdot \text{polarity}_n \;+\; \varepsilon, \quad \varepsilon \sim \mathcal N(0, 0.3).$$

A note that is universally high-quality gets endorsed by everyone regardless of polarity. A polarised note (high $|\delta_n|$) gets endorsed only by raters with matching polarity. The bridging algorithm should distinguish the two.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import spearmanr

np.random.seed(42)

N_RATERS = 200
N_NOTES = 60
RATINGS_PER_USER = 30
NOISE = 0.3

# Ground-truth latent variables
true_polarity_user = np.random.normal(0, 0.5, N_RATERS)
true_polarity_note = np.random.uniform(-1, 1, N_NOTES)
true_quality_note = np.random.normal(0, 0.5, N_NOTES)

# Each rater rates a sparse subset of notes
def generate_ratings(rater_ids, user_polarity):
    ratings = {}
    for u in rater_ids:
        notes_seen = np.random.choice(N_NOTES, RATINGS_PER_USER, replace=False)
        for i in notes_seen:
            r = (true_quality_note[i]
                 + user_polarity[u] * true_polarity_note[i]
                 + np.random.normal(0, NOISE))
            ratings[(u, i)] = r
    return ratings

ratings = generate_ratings(np.arange(N_RATERS), true_polarity_user)
print(f"Generated {len(ratings):,} ratings ({100*len(ratings)/(N_RATERS*N_NOTES):.0f}% density)")


## §3 — Implementing bridging matrix factorisation

We fit
$$r_{u,n} \;\approx\; \alpha_u + \beta_n + \gamma_u \cdot \delta_n$$
by minimising squared error plus an L2 penalty on all four parameter vectors. L-BFGS handles the joint optimisation in seconds at this scale (520 parameters, 6 000 ratings).

The key output for the algorithm is $\beta$ — the per-note helpfulness intercept. A note is "displayed" iff $\beta_n$ exceeds threshold.


In [ ]:
def fit_bridging(ratings, n_users, n_notes, lam=0.05, seed=0):
    '''Fit r_un = a_u + b_n + g_u * d_n by L-BFGS. Returns (alpha, beta, gamma, delta).'''
    rng = np.random.default_rng(seed)
    pairs = list(ratings.keys())
    rs = np.array([ratings[k] for k in pairs])
    us = np.array([k[0] for k in pairs])
    ns = np.array([k[1] for k in pairs])

    def unpack(theta):
        a = theta[:n_users]
        b = theta[n_users:n_users+n_notes]
        g = theta[n_users+n_notes:2*n_users+n_notes]
        d = theta[2*n_users+n_notes:]
        return a, b, g, d

    def neg_objective(theta):
        a, b, g, d = unpack(theta)
        pred = a[us] + b[ns] + g[us] * d[ns]
        sse = ((rs - pred) ** 2).sum()
        reg = lam * (a @ a + b @ b + g @ g + d @ d)
        return sse + reg

    theta0 = 0.01 * rng.standard_normal(2 * (n_users + n_notes))
    res = minimize(neg_objective, theta0, method="L-BFGS-B", options={"maxiter": 500})
    return unpack(res.x)

alpha, beta, gamma, delta = fit_bridging(ratings, N_RATERS, N_NOTES)

# How well did we recover ground truth?
print(f"Spearman(estimated note quality, true quality) = {spearmanr(beta, true_quality_note)[0]:.3f}")
print(f"Spearman(|estimated note polarity|, |true|)    = {spearmanr(np.abs(delta), np.abs(true_polarity_note))[0]:.3f}")


## §4 — Initial display: which notes pass the threshold?

We use the 60th percentile of $\beta$ as the "helpful" threshold (so ~40% of notes get displayed). This is a teaching simplification — the real Community Notes threshold is fixed at ~0.4 on a calibrated scale.


In [ ]:
THRESHOLD = np.quantile(beta, 0.6)
displayed_initial = np.where(beta > THRESHOLD)[0]
print(f"{len(displayed_initial)} of {N_NOTES} notes displayed (β > {THRESHOLD:.2f})")

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#0F6E56" if i in displayed_initial else "#9a9a94" for i in range(N_NOTES)]
ax.scatter(delta, beta, s=60, c=colors, edgecolor="white", linewidth=1.5)
ax.axhline(THRESHOLD, color="#993C1D", linestyle="--", linewidth=1, label=f"display threshold (β > {THRESHOLD:.2f})")
ax.set_xlabel("estimated note polarity δ")
ax.set_ylabel("estimated helpfulness intercept β")
ax.set_title("Bridging algorithm output: which notes are displayed?")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## §5 — Experiment A (Chuai): the consensus is unstable

**The mechanism Chuai et al. document.** Before display, a note is rated by a *self-selecting bridging-friendly* pool (people who go out of their way to find new notes). Once displayed, it reaches a much larger and more polarised audience. Their new ratings perturb the helpfulness intercept downward.

**Toy version.** We add 100 *polarised* raters drawn from a bimodal distribution at $\pm 1$. They mostly rate the *currently displayed* notes (display attracts attention). We refit the algorithm with the augmented rating set.


In [ ]:
# 100 new polarised raters drawn from bimodal {-1, +1}
N_NEW = 100
new_polarity = np.random.choice([-1, 1], N_NEW) + np.random.normal(0, 0.1, N_NEW)
new_idx = np.arange(N_RATERS, N_RATERS + N_NEW)

# Each new rater concentrates 70% of activity on currently-displayed notes
ratings_post = dict(ratings)
not_displayed = [n for n in range(N_NOTES) if n not in displayed_initial]
for u in new_idx:
    n_disp = int(RATINGS_PER_USER * 0.7)
    n_other = RATINGS_PER_USER - n_disp
    notes_disp = np.random.choice(displayed_initial, min(n_disp, len(displayed_initial)), replace=False)
    notes_other = np.random.choice(not_displayed, n_other, replace=False) if not_displayed else np.array([], dtype=int)
    for i in np.concatenate([notes_disp, notes_other]):
        r = (true_quality_note[i]
             + new_polarity[u - N_RATERS] * true_polarity_note[i]
             + np.random.normal(0, NOISE))
        ratings_post[(u, i)] = r

print(f"Pre-display:  {len(ratings):,} ratings, {N_RATERS} raters")
print(f"Post-display: {len(ratings_post):,} ratings, {N_RATERS + N_NEW} raters (added {N_NEW} polarised raters)")

alpha2, beta2, gamma2, delta2 = fit_bridging(ratings_post, N_RATERS + N_NEW, N_NOTES)
displayed_post = np.where(beta2 > THRESHOLD)[0]
lost = sorted(set(displayed_initial) - set(displayed_post))
kept = sorted(set(displayed_initial) & set(displayed_post))

print()
print(f"  initially displayed:           {len(displayed_initial)} notes")
print(f"  still displayed post-display:  {len(kept)}")
print(f"  LOST helpful status:           {len(lost)}  ({100*len(lost)/len(displayed_initial):.0f}%)")


In [ ]:
# Visualise: helpfulness intercept before vs after
fig, ax = plt.subplots(figsize=(9, 4.5))
order = np.argsort(beta)
x = np.arange(N_NOTES)
colors_before = ["#0F6E56" if n in displayed_initial else "#9a9a94" for n in order]
colors_after = ["#993C1D" if n in lost else
                ("#0F6E56" if n in displayed_post else "#9a9a94") for n in order]
ax.scatter(x - 0.15, beta[order], s=50, c=colors_before, label="before", marker="o", edgecolor="white", linewidth=1)
ax.scatter(x + 0.15, beta2[order], s=50, c=colors_after, label="after polarised post-display ratings", marker="s", edgecolor="white", linewidth=1)
ax.axhline(THRESHOLD, color="#993C1D", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlabel("note (sorted by initial β)")
ax.set_ylabel("helpfulness intercept β")
ax.set_title("Chuai's effect: post-display polarisation drags β below threshold")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Reading the plot.** Squares are post-display values; circles are pre-display. **Red squares** mark notes that lost helpful status — the polarised post-display ratings dragged their $\beta$ below threshold.

The toy produces a smaller effect (~10%) than the real-data finding (Chuai et al. report **30.2 %**). That's actually the right lesson: **bridging is robust by design** — the polarity term $\gamma_u \cdot \delta_n$ absorbs partisan disagreement, so adding more polarised raters doesn't, by itself, shift the helpfulness intercept much. The *direction* of the effect is reproducible at toy scale; the *magnitude* observed in real CN comes from factors this toy doesn't model — time dynamics, fixed calibrated thresholds, multidimensional polarity, partisan asymmetry in *which notes* attract post-display attention.

**The deeper point.** Classical aggregation theory assumes preferences are *exogenous to the rule*. On a live platform they are not — display changes who shows up to rate, and how. This is what we mean by *"deployed rules perturb their own inputs"*. Bridging mitigates the problem; it doesn't eliminate it.


## §6 — Experiment B (Nudo): the hyperactive minority

**The empirical claim.** Community Notes activity is *heavy-tailed* (a small minority generates most ratings) **and** *politically selective* (that minority is not representative of the broader rater pool). The two together break the implicit *universal participation* assumption of classical voting theory.

**Toy version.** Sample rater activity from a heavy-tailed distribution that is also asymmetric: raters with positive polarity get up to 8× the activity of raters with negative polarity. (Empirically this asymmetry runs in different directions depending on the topic / platform; we use a fixed-direction toy for clarity.)


In [ ]:
np.random.seed(7)  # fresh seed for clarity of the asymmetry pattern

# Heavy-tail (Pareto α=0.7) × asymmetric polarity boost (8× for positive polarity)
base_activity = np.random.pareto(0.7, N_RATERS) + 0.1
asym_boost = np.where(true_polarity_user > 0, 8.0, 1.0)
activity = base_activity * asym_boost

# Convert to ratings-per-user, normalised to ~6000 total ratings
ratings_per_user = np.clip(np.round(15 * activity / activity.mean()).astype(int), 1, N_NOTES)

# Top 5% by activity
top5_idx = np.argsort(-ratings_per_user)[:int(0.05 * N_RATERS)]

print(f"Total activity: {ratings_per_user.sum():,} ratings")
print(f"Top 5% (n={len(top5_idx)}) accounts for {100*ratings_per_user[top5_idx].sum()/ratings_per_user.sum():.0f}% of all ratings")
print()
print(f"Mean polarity:")
print(f"  full pool:    {true_polarity_user.mean():+.2f}")
print(f"  top 5% only:  {true_polarity_user[top5_idx].mean():+.2f}")
print(f"  bottom 95%:   {np.delete(true_polarity_user, top5_idx).mean():+.2f}")


In [ ]:
# Two side-by-side plots: activity distribution + polarity by activity rank
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# (a) Lorenz-style cumulative-share curve
sorted_act = np.sort(ratings_per_user)[::-1]  # high to low
cum_share = np.cumsum(sorted_act) / sorted_act.sum() * 100
pct = 100 * np.arange(1, len(sorted_act) + 1) / len(sorted_act)
axes[0].plot(pct, cum_share, color="#993C1D", linewidth=2)
axes[0].plot([0, 100], [0, 100], "--", color="#9a9a94", linewidth=1, label="equal participation")
axes[0].fill_between(pct, cum_share, pct, alpha=0.15, color="#993C1D")
axes[0].set_xlabel("rater percentile (most active → least active)")
axes[0].set_ylabel("cumulative share of ratings (%)")
axes[0].set_title("(a) Heavy-tailed activity")
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

# (b) Polarity distribution: full pool vs top 5%
axes[1].hist(true_polarity_user, bins=20, alpha=0.5, color="#9a9a94", label="full pool (200 raters)", density=True)
axes[1].hist(true_polarity_user[top5_idx], bins=10, alpha=0.7, color="#993C1D", label=f"top 5% by activity (n={len(top5_idx)})", density=True)
axes[1].axvline(0, color="#5a5a56", linestyle=":", linewidth=1)
axes[1].set_xlabel("rater polarity (hidden ground truth)")
axes[1].set_ylabel("density")
axes[1].set_title("(b) Top 5% is politically asymmetric")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


**Reading the plots.**

- *(a) Heavy-tailed activity.* The most-active 20 % of raters generate the bulk of ratings. The gap between the curve and the diagonal is what an inequality-of-influence diagnostic would call a *Gini area*: it measures how concentrated participation is.
- *(b) Politically asymmetric top 5 %.* The full-pool polarity distribution is centred near zero — most raters are moderate. The top-5 % distribution sits noticeably to the right (mean polarity +0.5 vs +0.0 in the full pool). The hyperactive minority is *not a random sample* of the rater population.

**Why this matters for bridging.** The algorithm fits a polarity axis from the rating pattern it observes. If the *observed* pattern over-represents one pole, the inferred polarity axis is biased — and the helpfulness intercept $\beta_n$ that bridging uses to display notes inherits that bias. The bridging property is only as good as the rater pool that feeds it.

**The deeper point.** Classical voting theorems implicitly assume *universal participation*: every voter votes. On a real platform, participation is heavy-tailed and politically asymmetric — and the "majority" the algorithm aggregates is the majority of *the participating subset*, not of the population.


## §7 — Takeaway

Both experiments expose what *classical aggregation theory assumes silently and is empirically false*.

1. **Chuai (§5).** Aggregation theory assumes preferences are *exogenous to the rule*. In deployment, displaying the rule's output changes who shows up to rate. Bridging consensus erodes after display because the post-display rater pool is more polarised than the pre-display pool.
2. **Nudo (§6).** Aggregation theory assumes *universal participation*. In deployment, participation is heavy-tailed and politically asymmetric. The bridging algorithm fits its polarity axis from a non-representative sample, and its outputs inherit the bias of the hyperactive minority.

These are not bugs in the bridging algorithm. They are *generic deployment conditions for any algorithmic aggregator that displays its outputs and relies on volunteer participation*. The lesson generalises beyond Community Notes — to any LLM training pipeline that solicits feedback from users who self-select into rating tasks.

## §8 — What to try next

- **Threshold sensitivity.** Re-run §5 with a stricter helpfulness threshold (e.g., `THRESHOLD = np.quantile(beta, 0.8)`). Does the loss-of-helpful-status rate go up?
- **Quality of bridging.** Compare bridging $\beta$ to a naive *mean rating* per note. Plot both against the ground-truth quality. Where does bridging beat the mean — and where does it not?
- **Heterogeneous polarity axes.** Replace 1-D polarity with 2-D. Does the bridging algorithm still recover quality cleanly when there are two orthogonal partisan axes?
- **Apply on real CN data.** X publishes daily Community Notes data dumps at [communitynotes.x.com/guide/en/under-the-hood/download-data](https://communitynotes.x.com/guide/en/under-the-hood/download-data). Implementing the full algorithm on real data is a meaty project — Twitter's open-source code is at [github.com/twitter/communitynotes](https://github.com/twitter/communitynotes).
